<a href="https://colab.research.google.com/github/Vigneshvar16/NLP_DSA/blob/main/NLP_ASSIGNMENT_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""DSA03 NLP - Assignment 2 (CO2): ONE program for Tasks 1-12.
N-gram language model + POS taggers for a transport-enquiry helpline.
Python standard library only (matplotlib is used only for the plots). Fixed seed -> reproducible.
Run:  python nlp_helpline.py      (writes results.txt and plots/*.png)
Student: Vigneshvar S (192424187)"""
import random, math, re, time, sys, os, itertools
from collections import Counter, defaultdict

SEED = 42
BOS, EOS, UNK = '<s>', '</s>', '<UNK>'
LOG = open('results.txt', 'w')
def P(*a):
    s = ' '.join(str(x) for x in a); print(s); LOG.write(s + '\n')
def H(t): P('\n' + '=' * 10, t, '=' * 10)
try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt; os.makedirs('plots', exist_ok=True)
except Exception:
    plt = None
def deep_size(o, seen=None):
    seen = seen if seen is not None else set()
    if id(o) in seen: return 0
    seen.add(id(o)); n = sys.getsizeof(o)
    if isinstance(o, dict): n += sum(deep_size(k, seen) + deep_size(v, seen) for k, v in o.items())
    elif isinstance(o, (list, tuple, set, frozenset)): n += sum(deep_size(x, seen) for x in o)
    elif hasattr(o, '__dict__'): n += deep_size(vars(o), seen)
    return n
PERF = {}   # Task 10: name -> dict(train_s, size_kb, quality, speed)

# ------------------------------------------------------------------ 0. CORPUS (synthetic, documented)
REAL = ("Chennai Madurai Vellore Salem Trichy Coimbatore Tirunelveli Erode Tiruppur Thanjavur Kanchipuram Nagercoil Karur "
        "Dindigul Hosur Ooty Cuddalore Namakkal Pollachi Kumbakonam Villupuram Tenkasi Theni Sivakasi Rameswaram Pudukkottai "
        "Ariyalur Perambalur Krishnagiri Dharmapuri Tiruvannamalai Nagapattinam Chidambaram Arakkonam Katpadi Gudiyatham Ambur "
        "Vaniyambadi Tindivanam").split()
def gen_names(n, rng, avoid, A, B, C):
    out = []
    while len(out) < n:
        nm = (rng.choice(A) + rng.choice(B) + rng.choice(C)).capitalize()
        if nm not in avoid and nm not in out: out.append(nm)
    return out
rng0 = random.Random(SEED)
OLD_LOCS = REAL + gen_names(960, rng0, set(REAL), "vel mad sal tir koi kan ero tan nag ram pud kum din ari pol oot cud kar nam sen ara mel kil pe".split(),
                            "a u i an ar al ur am ath".split(), "ore ai ur kal pet pattu nad ram gudi ipatti palli mangalam".split())
NEW_LOCS = gen_names(150, rng0, set(OLD_LOCS), "kotta pala vila sathy ana mett ela nava ponn thiru".split(),
                     "ya ri ku tha la mu".split(), "puram halli kottai vilai palayam nagar kadu mpatti".split())
ZIPF_W = lambda n, s=1.2: [1 / (r + 1) ** s for r in range(n)]
NUMW = "two three four five six seven eight nine ten".split()
TIMES = [f"{h}:{m:02d}" for h in range(5, 12) for m in (0, 15, 30, 45)]
DAYS = "Monday Tuesday Wednesday Thursday Friday Saturday Sunday".split()
TRAINS = "Cheran Pandian Vaigai Nilgiri Kovai Rockfort".split()
CMW = "enna ena eppo epo irukka iruka venum venam sollunga irukku".split()
LEM = {"tickets": "ticket", "seats": "seat", "booking": "book", "returning": "return", "changing": "change",
       "checking": "check", "cancelled": "cancel", "leaves": "leave"}

def S(*parts):
    out = []
    for p in parts:
        if isinstance(p, str):
            for x in p.split(): w, t = x.rsplit('/', 1); out.append((w, t))
        elif isinstance(p, tuple): out.append(p)
        else: out.extend(p)
    return out

def make_utt(rng, locs, wts, new=False):
    L = lambda: ((lambda w: w.lower() if rng.random() < 0.10 else w)(rng.choices(locs, wts)[0]), 'LOC')
    N = lambda: (rng.choice(NUMW), 'NUM')
    Tm = lambda: (rng.choice(TIMES), 'TIME')
    D = lambda: (rng.choice(DAYS), 'NNP')
    B = lambda: ((rng.choice(["bus", "train", "metro", "intercity", "bus"]) if new else "bus"), 'NN')
    VB = lambda *ws: (rng.choice(ws), 'VB')
    nn = lambda *ws: (rng.choice(ws), 'NN')
    T = [
        lambda: S("I/PRP want/VBP to/TO", VB("book", "return", "change", "cancel"), N(), "tickets/NNS to/TO", L()),
        lambda: S("what/WP is/VBZ the/DT fare/NN from/IN", L(), "to/TO", L()),
        lambda: S("when/WRB does/VBZ the/DT", B(), "to/TO", L(), "leave/VB"),
        lambda: S("is/VBZ there/RB a/DT", B(), "from/IN", L(), "to/TO", L(), "on/IN", D()),
        lambda: S("I/PRP want/VBP to/TO", VB("cancel", "return", "change", "check"), "my/PRP$ ticket/NN"),
        lambda: S("please/UH book/VB a/DT seat/NN on/IN the/DT", Tm(), B(), "to/TO", L()),
        lambda: S("how/WRB much/JJ is/VBZ the/DT", nn("return", "season", "express", "change"), "fare/NN"),
        lambda: S("I/PRP will/MD", VB("return", "book", "check", "change"), "on/IN", D()),
        lambda: S("where/WRB is/VBZ the/DT bus/NN", nn("stand", "stop")),
        lambda: S("does/VBZ the/DT", (rng.choice(TRAINS), 'NNP'), "express/NN stop/VB at/IN", L()),
        lambda: S("my/PRP$", nn("booking", "return", "change", "check"), "was/VBD cancelled/VBN"),
        lambda: S("I/PRP am/VBP", (rng.choice(["booking", "returning", "changing", "checking"]), 'VBG'), N(), "tickets/NNS"),
        lambda: S("can/MD I/PRP get/VB a/DT", nn("refund", "return", "change"), "for/IN my/PRP$ ticket/NN"),
        lambda: S("please/UH", VB("refund", "cancel"), "my/PRP$ money/NN"),
        lambda: S("the/DT", B(), "to/TO", L(), "was/VBD late/JJ"),
        lambda: S("is/VBZ there/RB a/DT", B(), "or/CC", B(), "to/TO", L()),
        lambda: S("the/DT first/JJ", B(), "leaves/VBZ at/IN", Tm()),
        lambda: S("the/DT next/JJ", nn("stop", "change", "return"), "is/VBZ", L()),
    ]
    WT = [3, 3, 2, 3, 2, 2, 1, 1, 1, 2, 1, 1, 2, 1, 2, 1, 1, 1]
    u = rng.choices(T, WT)[0]()
    if rng.random() < 0.22: u = [(rng.choice(["hello", "hi", "uh", "um"]), 'UH')] + u
    if new and rng.random() < 0.8: u = u + [(rng.choice(CMW), 'CM')]
    return u

def group_split(utts, rng, fr=(0.8, 0.1, 0.1)):
    """every distinct utterance string goes to exactly ONE split (no leakage between splits)"""
    groups = defaultdict(list)
    for u in utts: groups[' '.join(w.lower() for w, _ in u)].append(u)
    keys = list(groups); rng.shuffle(keys)
    targ = [f * len(utts) for f in fr]; fill = [0, 0, 0]; out = [[], [], []]
    for k in keys:
        j = min(range(3), key=lambda j: fill[j] / targ[j]); out[j].extend(groups[k]); fill[j] += len(groups[k])
    for o in out: rng.shuffle(o)
    return out

rng = random.Random(SEED)
ALL = [make_utt(rng, OLD_LOCS, ZIPF_W(len(OLD_LOCS))) for _ in range(24000)]
TRAIN, DEV, TEST = group_split(ALL, random.Random(SEED))
lm = lambda u: [w.lower() for w, _ in u]
train_lm, dev_lm, test_lm = [lm(u) for u in TRAIN], [lm(u) for u in DEV], [lm(u) for u in TEST]
assert not (set(map(tuple, train_lm)) & set(map(tuple, dev_lm))) and not (set(map(tuple, train_lm)) & set(map(tuple, test_lm)))

# ------------------------------------------------------------------ TASK 1
H('TASK 1 - counting words, Zipf, vocabulary')
tok = [w for u in ALL for w, _ in u]; NTOK = len(tok)
types = Counter(w.lower() for w in tok); forms = set(tok); lemmas = {LEM.get(w, w) for w in types}
P(f'corpus: {len(ALL)} utterances, {NTOK} tokens | split {len(TRAIN)}/{len(DEV)}/{len(TEST)} utterances ({sum(map(len,TRAIN))}/{sum(map(len,DEV))}/{sum(map(len,TEST))} tokens)')
P(f'tokens={NTOK}  word forms (case-sensitive)={len(forms)}  types (case-folded)={len(types)}  lemmas={len(lemmas)}  (<s>,</s> added per utterance, not counted as words)')
istime = lambda w: re.fullmatch(r'\d+:\d+', w) is not None
split_vocab = set(); [split_vocab.update(re.split(r'(:)', w) if istime(w) else [w]) for w in types]
ntime = sum(1 for w in tok if istime(w))
P(f'decision 1 case-folding: vocabulary {len(forms)} -> {len(types)} types | decision 2 "6:30" as 1 token vs 3 tokens: vocabulary {len(types)} vs {len(split_vocab)}, tokens {NTOK} vs {NTOK + 2*ntime}')
fr = sorted(types.values(), reverse=True)
xs = [math.log(r + 1) for r in range(len(fr))]; ys = [math.log(f) for f in fr]
mx, my = sum(xs) / len(xs), sum(ys) / len(ys)
slope = sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / sum((x - mx) ** 2 for x in xs); icpt = my - slope * mx
r2 = 1 - sum((y - (icpt + slope * x)) ** 2 for x, y in zip(xs, ys)) / sum((y - my) ** 2 for y in ys)
P(f'Zipf: log f = {icpt:.2f} {slope:+.2f} log(rank), R^2={r2:.3f} (ideal slope -1)')
ranked = [w for w, _ in types.most_common()]
res = sorted(((ys[i] - (icpt + slope * xs[i]), ranked[i], i + 1, fr[i]) for i in range(len(fr))))
P('largest deviations (residual, word, rank, freq): below line', [(round(a, 2), b, c, d) for a, b, c, d in res[:3]], '| above line', [(round(a, 2), b, c, d) for a, b, c, d in res[-3:]])
if plt:
    plt.figure(figsize=(5, 3.6)); plt.loglog(range(1, len(fr) + 1), fr, '.', label='corpus'); plt.loglog(range(1, len(fr) + 1), [math.exp(icpt + slope * x) for x in xs], label=f'fit slope {slope:.2f}')
    plt.xlabel('rank'); plt.ylabel('frequency'); plt.title('Zipf plot'); plt.legend(); plt.tight_layout(); plt.savefig('plots/zipf.png', dpi=130); plt.close()
trc = Counter(w for s in train_lm for w in s); NTR = sum(trc.values())
build_vocab = lambda t: {w for w, c in trc.items() if c > t}
for t in (1, 2, 3, 5):
    Vt = build_vocab(t); unk = sum(c for w, c in trc.items() if c <= t)
    P(f'threshold {t} (count<={t} -> <UNK>): vocabulary={len(Vt)+1} types, {100*unk/NTR:.2f}% of training tokens become <UNK>')
dev_oov_full = sum(1 for s in dev_lm for w in s if w not in trc) / sum(map(len, dev_lm))
VOC = build_vocab(1)
dev_oov_t1 = sum(1 for s in dev_lm for w in s if w not in VOC) / sum(map(len, dev_lm))
P(f'OOV rate of dev set: {100*dev_oov_full:.2f}% against the full training vocabulary, {100*dev_oov_t1:.2f}% against the threshold-1 vocabulary (used from here on)')
mapu = lambda s, V: [w if w in V else UNK for w in s]
train_m, dev_m, test_m = [mapu(s, VOC) for s in train_lm], [mapu(s, VOC) for s in dev_lm], [mapu(s, VOC) for s in test_lm]

# ------------------------------------------------------------------ n-gram machinery
class Counts:
    def __init__(s, sents):
        s.u, s.b, s.t = Counter(), Counter(), Counter()
        for x in sents:
            q = [BOS, BOS] + x + [EOS]
            for i in range(2, len(q)):
                s.u[q[i]] += 1; s.b[(q[i - 1], q[i])] += 1; s.t[(q[i - 2], q[i - 1], q[i])] += 1
        s.N = sum(s.u.values()); s.c2 = Counter(); s.c3 = Counter()
        for (v, w), c in s.b.items(): s.c2[v] += c
        for (u, v, w), c in s.t.items(): s.c3[(u, v)] += c
        s.V = len(s.u)                       # predictable symbols (words, <UNK>, </s>)
class MLE:
    def __init__(s, C, n): s.C, s.n = C, n
    def p(s, w, u, v):
        C = s.C
        if s.n == 1: return C.u[w] / C.N
        if s.n == 2: d = C.c2[v]; return C.b[(v, w)] / d if d else 0.0
        d = C.c3[(u, v)]; return C.t[(u, v, w)] / d if d else 0.0
class AddK:
    def __init__(s, C, n, k): s.C, s.n, s.k = C, n, k
    def p(s, w, u, v):
        C, k = s.C, s.k
        if s.n == 2: return (C.b[(v, w)] + k) / (C.c2[v] + k * C.V)
        return (C.t[(u, v, w)] + k) / (C.c3[(u, v)] + k * C.V)
def gt_discounts(cnt, K=5):
    Nr = Counter(cnt.values()); A = (K + 1) * Nr[K + 1] / Nr[1]; d = {}
    for r in range(1, K + 1):
        d[r] = min(1.0, max(0.05, (((r + 1) * Nr[r + 1] / (r * Nr[r])) - A) / (1 - A))) if Nr[r] and Nr[r + 1] else 1.0
    return d
class Katz:
    """Katz backoff with Good-Turing discounting; order 2 = bigram->unigram, order 3 = trigram->bigram->unigram"""
    def __init__(s, C, order=3):
        s.C, s.order = C, order; s.d2, s.d3 = gt_discounts(C.b), gt_discounts(C.t)
        s.nx2, s.nx3 = defaultdict(list), defaultdict(list)
        for (v, w) in C.b: s.nx2[v].append(w)
        for (u, v, w) in C.t: s.nx3[(u, v)].append(w)
        s.a2, s.a3 = {}, {}
    def p1(s, w): return s.C.u[w] / s.C.N
    def p2(s, w, v):
        C = s.C; c = C.b.get((v, w), 0)
        if c: s.alpha2(v); return s.a2[v][1] * s.d2.get(c, 1.0) * c / C.c2[v]
        return s.alpha2(v) * s.p1(w) if C.c2[v] else s.p1(w)
    def _res(s, S, c):                         # reserved mass; keep a floor so the model never returns 0
        eps = min(0.5, 1.0 / (c + 1)); r = 1 - S
        return (r, 1.0) if r >= eps else (eps, (1 - eps) / S)
    def alpha2(s, v):
        if v not in s.a2:
            C = s.C; sn = s.nx2[v]
            S = sum(s.d2.get(C.b[(v, w)], 1.0) * C.b[(v, w)] / C.c2[v] for w in sn); den = 1 - sum(s.p1(w) for w in sn)
            r, f = s._res(S, C.c2[v]); s.a2[v] = (r / den if den > 1e-12 else 0.0, f)
        return s.a2[v][0]
    def alpha3(s, u, v):
        if (u, v) not in s.a3:
            C = s.C; sn = s.nx3[(u, v)]
            S = sum(s.d3.get(C.t[(u, v, w)], 1.0) * C.t[(u, v, w)] / C.c3[(u, v)] for w in sn); den = 1 - sum(s.p2(w, v) for w in sn)
            r, f = s._res(S, C.c3[(u, v)]); s.a3[(u, v)] = (r / den if den > 1e-12 else 0.0, f)
        return s.a3[(u, v)][0]
    def p(s, w, u, v):
        if s.order == 1: return s.p1(w)
        if s.order == 2: return s.p2(w, v)
        C = s.C; c = C.t.get((u, v, w), 0)
        if c: s.alpha3(u, v); return s.a3[(u, v)][1] * s.d3.get(c, 1.0) * c / C.c3[(u, v)]
        return s.alpha3(u, v) * s.p2(w, v) if C.c3[(u, v)] else s.p2(w, v)
class Stupid:
    def __init__(s, C, a=0.4): s.C, s.a = C, a
    def p(s, w, u, v):
        C = s.C
        if C.t[(u, v, w)]: return C.t[(u, v, w)] / C.c3[(u, v)]
        if C.b[(v, w)]: return s.a * C.b[(v, w)] / C.c2[v]
        return s.a * s.a * C.u[w] / C.N
band = lambda c: 0 if c == 0 else 1 if c <= 2 else 2 if c <= 9 else 3
class Interp:
    def __init__(s, C, lam): s.C, s.lam = C, lam
    def p(s, w, u, v):
        C = s.C; c3 = C.c3[(u, v)]; l3, l2, l1 = s.lam[band(c3)] if s.lam else (0.6, 0.3, 0.1)
        return (l3 * C.t[(u, v, w)] / c3 if c3 else 0) + (l2 * C.b[(v, w)] / C.c2[v] if C.c2[v] else 0) + l1 * C.u[w] / C.N
def em_lambdas(Cr, held, iters=40):
    lam = {b: [1 / 3] * 3 if b else [0, .5, .5] for b in range(4)}; cnt = Counter()
    for _ in range(iters):
        acc = {b: [0, 0, 0] for b in range(4)}; cnt = Counter()
        for x in held:
            q = [BOS, BOS] + x + [EOS]
            for i in range(2, len(q)):
                u, v, w = q[i - 2], q[i - 1], q[i]; c3 = Cr.c3[(u, v)]; b = band(c3)
                pr = [Cr.t[(u, v, w)] / c3 if c3 else 0, Cr.b[(v, w)] / Cr.c2[v] if Cr.c2[v] else 0, Cr.u[w] / Cr.N]
                comp = [lam[b][k] * pr[k] for k in range(3)]; z = sum(comp)
                if z == 0: continue
                for k in range(3): acc[b][k] += comp[k] / z
                cnt[b] += 1
        for b in range(4):
            if cnt[b]: lam[b] = [a / cnt[b] for a in acc[b]]
    return lam, cnt
def xent(model, sents):
    bits = n = zeros = 0
    for x in sents:
        q = [BOS, BOS] + x + [EOS]
        for i in range(2, len(q)):
            p = model.p(q[i], q[i - 2], q[i - 1]); n += 1
            if p <= 0: zeros += 1
            else: bits -= math.log2(p)
    return (bits / n if not zeros else math.inf), n, zeros, bits / len(sents)
ppl = lambda h: 2 ** h if h != math.inf else math.inf
def timed(name, fn, quality=None):
    t0 = time.time(); m = fn(); PERF.setdefault(name, {})['train_s'] = time.time() - t0; PERF[name]['size_kb'] = deep_size(m) / 1024; return m

# ------------------------------------------------------------------ TASK 2
H('TASK 2 - unsmoothed n-grams')
C = timed('counts (uni/bi/tri)', lambda: Counts(train_m)); V = C.V
uni, bi, tri = MLE(C, 1), MLE(C, 2), MLE(C, 3)
P('chain rule: P(w1..wn)=prod P(wi|w1..wi-1); bigram assumes P(wi|w1..wi-1)=P(wi|wi-1); trigram assumes P(wi|wi-2,wi-1). MLE: P(w|v)=c(v,w)/c(v)')
def hand(sent, n=2):
    q = [BOS, BOS] + sent + [EOS]; prod = 1.0; fac = []
    for i in range(2, len(q)):
        if n == 2: c, d = C.b[(q[i - 1], q[i])], C.c2[q[i - 1]]; lab = f'P({q[i]}|{q[i-1]})'
        else: c, d = C.t[(q[i - 2], q[i - 1], q[i])], C.c3[(q[i - 2], q[i - 1])]; lab = f'P({q[i]}|{q[i-2]},{q[i-1]})'
        pr = c / d if d else 0.0; prod *= pr; fac.append(f'{lab}={c}/{d}={pr:.4f}')
    return prod, fac
for sent in (['i', 'want', 'to', 'book', 'two', 'tickets', 'to', 'vellore'], ['where', 'is', 'the', 'bus', 'stand'], ['please', 'cancel', 'my', 'ticket']):
    pr, fac = hand(sent); P('bigram', ' '.join(sent), '->', f'{pr:.3e}'); P('   ' + ' x '.join(fac))
zero_utt = next(s for s in dev_m if hand(s)[0] == 0)
q = [BOS] + zero_utt + [EOS]; zb = next((q[i], q[i + 1]) for i in range(len(q) - 1) if C.b[(q[i], q[i + 1])] == 0)
P('zero-probability utterance (dev):', ' '.join(zero_utt), '| responsible unseen bigram:', zb, '| bigram probability =', hand(zero_utt)[0])
ZERO_UTT = zero_utt
poss2, poss3 = (V + 1) * V, (V + 1) * V * V
dev_b = [(a, b) for x in dev_m for a, b in zip([BOS] + x, x + [EOS])]; dev_t = [t for x in dev_m for t in zip([BOS, BOS] + x, [BOS] + x + [EOS], x + [EOS])]
P(f'sparsity: observed bigrams {len(C.b)}/{poss2} = {100*len(C.b)/poss2:.2f}% ; observed trigrams {len(C.t)}/{poss3} = {100*len(C.t)/poss3:.3f}%')
P(f'dev bigrams never seen in training: {100*sum(1 for b in dev_b if b not in C.b)/len(dev_b):.2f}% ; dev trigrams never seen: {100*sum(1 for t in dev_t if t not in C.t)/len(dev_t):.2f}%')
gr = random.Random(SEED)
NX2, NX3 = defaultdict(list), defaultdict(list)
for (v, w), c in C.b.items(): NX2[v].append((w, c))
for (u, v, w), c in C.t.items(): NX3[(u, v)].append((w, c))
def sample(n):
    out = []
    for _ in range(10):
        s = [BOS, BOS]
        while s[-1] != EOS and len(s) < 22:
            cand = list(C.u.items()) if n == 1 else NX2[s[-1]] if n == 2 else NX3[(s[-2], s[-1])]
            s.append(gr.choices([w for w, _ in cand], [c for _, c in cand])[0])
        out.append(' '.join(s[2:]))
    return out
for n, name in ((1, 'unigram'), (2, 'bigram'), (3, 'trigram')):
    P(f'{name} samples:', ' | '.join(sample(n)[:4]))

# ------------------------------------------------------------------ TASK 3
H('TASK 3 - smoothing and discounting')
ctx_w = 'to'; cw = sorted(((w, C.b[(ctx_w, w)]) for w in C.u if C.b[(ctx_w, w)]), key=lambda x: -x[1]); Ch = C.c2[ctx_w]
P(f'context "{ctx_w}": c({ctx_w})={Ch}, V={V}, seen followers={len(cw)}  (add-one: c*=(c+1)*c({ctx_w})/(c({ctx_w})+V))')
P('word      count  MLE-p     add1-c*   add1-p    add-k(0.05)-p')
k_show = 0.05
for w, c in cw[:5]: P(f'{w:10s}{c:6d}  {c/Ch:.5f}  {(c+1)*Ch/(Ch+V):8.2f}  {(c+1)/(Ch+V):.5f}  {(c+k_show)/(Ch+k_show*V):.5f}')
seen_p1 = sum((c + 1) / (Ch + V) for _, c in cw)
P(f'mass taken from the {len(cw)} seen bigrams by add-one: {1-seen_p1:.4f} (their MLE mass was 1.0000); each of the {V-len(cw)} unseen followers gets {1/(Ch+V):.6f}')
top_w, top_c = cw[0]; P(f'severity: top follower "{top_w}" keeps {100*((top_c+1)/(Ch+V))/(top_c/Ch):.1f}% of its MLE probability, because V={V} extra pseudo-counts are added to a context seen only {Ch} times')
Nr = Counter(C.b.values()); N2 = sum(C.b.values()); N0 = poss2 - len(C.b)
P('Good-Turing bigram table  r: Nr -> c*=(r+1)N(r+1)/Nr :', '; '.join(f'{r}: {Nr[r]} -> {(r+1)*Nr[r+1]/Nr[r]:.2f}' for r in range(1, 8)))
P(f'unseen bigram mass P0=N1/N={Nr[1]}/{N2}={Nr[1]/N2:.5f}; N0={N0}; each unseen bigram c*0=N1/N0={Nr[1]/N0:.2e}, p={Nr[1]/N0/N2:.2e}; counts above 5 are kept (c*=c) because Nr gets noisy/zero')
ks = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]; kres = {k: ppl(xent(AddK(C, 2, k), dev_m)[0]) for k in ks}; bestk = min(kres, key=kres.get)
P('add-k dev perplexity (bigram):', {k: round(v, 2) for k, v in kres.items()}, '-> best k =', bestk, '(tuned on DEV only)')

# ------------------------------------------------------------------ TASK 4
H('TASK 4 - backoff and deleted interpolation')
katz3 = timed('Katz trigram (GT)', lambda: Katz(C, 3)); katz2 = Katz(C, 2); stupid = Stupid(C)
sh = random.Random(SEED); idx = list(range(len(train_m))); sh.shuffle(idx); cut = int(0.9 * len(idx))
Cr = Counts([train_m[i] for i in idx[:cut]]); held = [train_m[i] for i in idx[cut:]]
lam, cnt = timed('deleted interpolation (EM)', lambda: em_lambdas(Cr, held))
names_b = {0: 'c(u,v)=0', 1: 'c(u,v)=1-2', 2: 'c(u,v)=3-9', 3: 'c(u,v)>=10'}
for b in range(4): P(f'lambda bands {names_b[b]:12s} (l3,l2,l1)=({lam[b][0]:.3f},{lam[b][1]:.3f},{lam[b][2]:.3f}) sum={sum(lam[b]):.3f}  (held-out tokens {cnt[b]})')
interp = Interp(C, lam); lin = Interp(C, None)
ex = next((u, v) for (u, v), c in C.c3.items() if c == 1 and C.c2[v] > 50)
P(f'worked example: context {ex} seen once -> band 1 puts {lam[1][0]:.2f} on trigram, {lam[1][1]:.2f} on bigram, {lam[1][2]:.2f} on unigram; a frequent context (band 3) puts {lam[3][0]:.2f} on the trigram')
for ctxs in [(BOS, BOS), ex]:
    s = sum(katz3.p(w, *ctxs) for w in C.u); s2 = sum(interp.p(w, *ctxs) for w in C.u); P(f'validity check context {ctxs}: sum_w Katz={s:.6f}, sum_w Interp={s2:.6f}')
for c_, nm in ((katz3, 'Katz'), (interp, 'Interp')):
    assert abs(sum(c_.p(w, *ex) for w in C.u) - 1) < 1e-6, nm + ' not a distribution'
tr = next(x for x in dev_m if any(C.t[(a, b, c)] == 0 and C.c3[(a, b)] > 0 and C.c2[c] >= 0 for a, b, c in zip([BOS, BOS] + x, [BOS] + x + [EOS], x + [EOS])))
qq = [BOS, BOS] + tr + [EOS]; P('trace utterance:', ' '.join(tr))
for i in range(2, len(qq)):
    u, v, w = qq[i - 2], qq[i - 1], qq[i]; unseen = C.t[(u, v, w)] == 0 and C.c3[(u, v)] > 0
    P(f'  P({w}|{u},{v}): trigram count={C.t[(u,v,w)]} Katz={katz3.p(w,u,v):.5f} Interp={interp.p(w,u,v):.5f}' + ('   <-- unseen trigram: Katz = alpha*P2 = %.3f*%.5f ; Interp mixes l3*0 + l2*P2 + l1*P1' % (katz3.alpha3(u, v), katz3.p2(w, v)) if unseen else ''))

# ------------------------------------------------------------------ TASK 5
H('TASK 5 - entropy, cross-entropy, perplexity')
ent = lambda d: -sum(p * math.log2(p) for p in d if p > 0)
cands = []
for v in C.c2:
    if C.c2[v] >= 200 and len([w for w in C.u if C.b[(v, w)]]) > 1: cands.append((ent([C.b[(v, w)] / C.c2[v] for w in C.u if C.b[(v, w)]]), v))
cands.sort(); picks = [cands[0], cands[len(cands) // 2], cands[-1]]
for h_, v in picks:
    top = sorted(((C.b[(v, w)] / C.c2[v], w) for w in C.u if C.b[(v, w)]), reverse=True)[:3]
    P(f'H(next word | "{v}") = {h_:.3f} bits over {sum(1 for w in C.u if C.b[(v,w)])} followers; top: ' + ', '.join(f'{w} {p:.2f}' for p, w in top) + f'  (uniform over them would be {math.log2(sum(1 for w in C.u if C.b[(v,w)])):.2f})')
models = [('unigram MLE', uni), ('bigram MLE', bi), ('trigram MLE', tri), ('bigram add-one', AddK(C, 2, 1.0)), (f'bigram add-k ({bestk})', AddK(C, 2, bestk)),
          ('bigram Katz/Good-Turing', katz2), ('trigram Katz backoff', katz3), ('trigram stupid backoff (score)', stupid),
          ('trigram linear interp (.6/.3/.1)', lin), ('trigram deleted interp', interp)]
P('\nmodel                              dev H(bits/word) dev PPL   test PPL   bits/utterance(dev)')
RES = {}
for nm, m in models:
    t0 = time.time(); hd, nd, zd, bu = xent(m, dev_m); ht, nt, zt, _ = xent(m, test_m); el = time.time() - t0
    RES[nm] = (ppl(hd), ppl(ht)); PERF.setdefault(nm, {}).update(quality=f'dev PPL {ppl(hd):.2f}', speed=f'{(nd+nt)/el:,.0f} tok/s')
    P(f'{nm:34s} {hd if hd!=math.inf else float("inf"):8.3f}   {ppl(hd):9.2f}  {ppl(ht):9.2f}   {bu:8.2f}' + (f'   ({zd} zero-prob events)' if zd else ''))
P('(perplexity 90 would mean the model is as unsure as if it chose among ~90 equally likely words at every step)')
# unfair comparison: different <UNK> treatment
VOC5 = build_vocab(100); C5 = Counts([mapu(s, VOC5) for s in train_lm]); dev5 = [mapu(s, VOC5) for s in dev_lm]
lam5, _ = em_lambdas(Counts([mapu(train_lm[i], VOC5) for i in idx[:cut]]), [mapu(train_lm[i], VOC5) for i in idx[cut:]])
pA, pB = ppl(xent(interp, dev_m)[0]), ppl(xent(Interp(C5, lam5), dev5)[0])
P(f'UNFAIR comparison: same interpolated model with <UNK> threshold 1 (|V|={V}) PPL={pA:.2f} vs threshold 100 (|V|={C5.V}) PPL={pB:.2f} -> "better" only because more words are hidden inside <UNK>')
# rescoring experiment
CONF = {'two': ['to', 'too'], 'to': ['two', 'too'], 'four': ['for'], 'for': ['four'], 'ticket': ['tickit'], 'bus': ['boss'], 'seat': ['seed', 'sit'],
        'book': ['look', 'back'], 'fare': ['fair'], 'return': ['retain'], 'want': ['wont'], 'the': ['a', 'that'], 'is': ['his'], 'on': ['in'], 'cancel': ['counsel'], 'my': ['me']}
LOCSET = {n.lower() for n in OLD_LOCS + NEW_LOCS}
def corrupt(ref, r):
    for _ in range(20):
        h = list(ref)
        for _ in range(r.choice([1, 1, 2])):
            p = r.randrange(len(h)); w = h[p]
            if w in CONF: h[p] = r.choice(CONF[w])
            elif w in LOCSET and len(w) > 4: j = r.randrange(1, len(w) - 1); h[p] = w[:j] + w[j + 1:]
        if h != ref: return h
    return ref[:-1] or ref
def make_sets(refs, r, n=200):
    sets = []
    for ref in r.sample(refs, n):
        hyps = [ref]; tries = 0
        while len(hyps) < 5:
            tries += 1; h = corrupt(ref, r)
            if tries > 12:                                   # few confusable words: add a deletion/insertion error
                h = list(ref); p = r.randrange(len(h) + 1)
                if r.random() < .5 and len(h) > 2: del h[min(p, len(h) - 1)]
                else: h.insert(p, r.choice(['a', 'the', 'to', 'uh']))
            if h not in hyps: hyps.append(h)
            if tries > 200: break
        r.shuffle(hyps); sets.append((ref, [(h, r.gauss(0, 0.5)) for h in hyps]))   # acoustic scores carry no information
    return sets
def edit(a, b):
    d = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(b) + 1): prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
    return d[len(b)]
def wer(sets, pick): return sum(edit(pick(h), ref) for ref, h in sets) / sum(len(ref) for ref, _ in sets)
def lm_logp(m, toks):
    q = [BOS, BOS] + mapu(toks, VOC) + [EOS]; return sum(math.log2(max(m.p(q[i], q[i - 2], q[i - 1]), 1e-12)) for i in range(2, len(q)))
def best_by(sets, scorer, w):
    return lambda hyps: max(hyps, key=lambda h: h[1] + w * scorer(h[0]))[0]
DEVSETS = make_sets(dev_lm, random.Random(SEED + 1)); TESTSETS = make_sets(test_lm, random.Random(SEED + 2))
w0 = wer(TESTSETS, lambda h: max(h, key=lambda x: x[1])[0])
P(f'\nrescoring: 200 test utterances x 5 hypotheses (reference included, acoustic scores random). WER of recogniser 1-best = {100*w0:.2f}%')
P('LM used for rescoring           LM weight(dev-tuned)  WER after   PPL(dev)')
RESC = {}
for nm in ('unigram MLE', 'bigram add-one', f'bigram add-k ({bestk})', 'trigram Katz backoff', 'trigram deleted interp'):
    m = dict(models)[nm]; sc = lambda t, m=m: lm_logp(m, t)
    bw = min((0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0), key=lambda w: wer(DEVSETS, best_by(DEVSETS, sc, w)))
    RESC[nm] = (bw, wer(TESTSETS, best_by(TESTSETS, sc, bw))); P(f'{nm:32s} {bw:8.2f}            {100*RESC[nm][1]:6.2f}%   {RES[nm][0]:8.2f}')

# ================================================================== PART 2: TAGGING (Tasks 6-9)
H('TASK 6 - word classes and tagset')
TAGSET = [('NN', 'common noun', 'bus, fare, stand', 'NN'), ('NNS', 'plural noun', 'tickets', 'NNS'), ('LOC', 'place / station name', 'Vellore', 'NNP'),
          ('NNP', 'other proper noun', 'Monday, Cheran', 'NNP'), ('NUM', 'number word', 'two', 'CD'), ('TIME', 'clock time', '6:30', 'CD'),
          ('VB', 'base verb', 'book', 'VB'), ('VBP', 'present verb', 'want, am', 'VBP'), ('VBZ', '3rd-sg present', 'is', 'VBZ'), ('VBD', 'past verb', 'was', 'VBD'),
          ('VBG', 'gerund / -ing', 'booking', 'VBG'), ('VBN', 'past participle', 'cancelled', 'VBN'), ('MD', 'modal', 'will, can', 'MD'),
          ('PRP', 'personal pronoun', 'I', 'PRP'), ('PRP$', 'possessive', 'my', 'PRP$'), ('DT', 'determiner', 'the, a', 'DT'), ('IN', 'preposition', 'from, on', 'IN'),
          ('TO', 'to', 'to', 'TO'), ('CC', 'conjunction', 'or', 'CC'), ('JJ', 'adjective', 'first, late', 'JJ'), ('RB', 'adverb / existential there', 'there', 'RB, EX'),
          ('WRB', 'wh-adverb', 'when, where', 'WRB'), ('WP', 'wh-pronoun', 'what', 'WP'), ('UH', 'interjection / filler', 'hello, please, um', 'UH')]
P(f'tagset: {len(TAGSET)} tags -> PTB mapping: ' + '; '.join(f'{t}->{p}' for t, _, _, p in TAGSET))
TAG_TRAIN, TAG_DEV, TAG_TEST = TRAIN[:5000], DEV[:1500], TEST
wt = defaultdict(Counter)
for u in ALL:
    for w, t in u: wt[w.lower()][t] += 1
closed = {t: Counter(w.lower() for u in ALL for w, tt in u if tt == t) for t in ('DT', 'IN', 'TO', 'CC', 'PRP', 'PRP$', 'MD', 'WRB', 'WP', 'UH')}
P('closed classes in corpus: ' + ' | '.join(f'{t}: {", ".join(sorted(c))}' for t, c in closed.items()))
amb_types = [w for w, c in wt.items() if len(c) > 1]; amb_tok = sum(sum(wt[w].values()) for w in amb_types)
P(f'ambiguity: {len(amb_types)}/{len(wt)} word types ({100*len(amb_types)/len(wt):.1f}%) and {100*amb_tok/NTOK:.1f}% of tokens have >1 tag; average tags per ambiguous token = {sum(len(wt[w])*sum(wt[w].values()) for w in amb_types)/amb_tok:.2f}; examples: ' + ', '.join(f'{w}{sorted(wt[w])}' for w in sorted(amb_types)[:6]))
sample_ann, nt_ = [], 0
for u in TAG_TRAIN:
    sample_ann += u; nt_ += len(u)
    if nt_ >= 3000: break
P(f'annotated sample: {len(sample_ann)} tokens; tag frequencies: ' + ', '.join(f'{t} {c}' for t, c in Counter(t for _, t in sample_ann).most_common(10)))
ar = random.Random(SEED); a1 = [t for _, t in sample_ann[:500]]; a2 = []
for (w, t) in sample_ann[:500]:           # SIMULATED second annotator (replace with a real second person's tags)
    alts = [x for x in wt[w.lower()] if x != t]
    a2.append(ar.choice(alts) if alts and ar.random() < 0.15 else (ar.choice([x[0] for x in TAGSET]) if ar.random() < 0.002 else t))
po = sum(x == y for x, y in zip(a1, a2)) / 500; c1, c2 = Counter(a1), Counter(a2); pe = sum(c1[t] * c2[t] for t in c1) / 500 ** 2
P(f'inter-annotator agreement on 500 doubly tagged tokens (SIMULATED annotator 2): raw={100*po:.1f}%  Cohen kappa={(po-pe)/(1-pe):.3f}')

# ---- shared tagger tools
dic = defaultdict(Counter)
for u in TAG_TRAIN:
    for w, t in u: dic[w.lower()][t] += 1
prior = Counter(t for u in TAG_TRAIN for _, t in u)
OPEN = {'NN', 'NNS', 'LOC', 'NNP', 'JJ', 'VBG', 'VBN', 'RB', 'VB'}
def unk_guess(w): return 'TIME' if istime(w) else 'LOC' if w[:1].isupper() else 'NN'
def make_mft(d): return lambda raw: [d[w.lower()].most_common(1)[0][0] if w.lower() in d else unk_guess(w) for w in raw]
mft_tag = make_mft(dic)
def evaluate(fn, sents, d):
    n = ok = na = a = nu = au = 0
    for u in sents:
        raw = [w for w, _ in u]; pred = fn(raw)
        for w, (_, g), p in zip(raw, u, pred):
            n += 1; ok += g == p; lw = w.lower()
            if lw not in d: nu += 1; au += g == p
            elif len(d[lw]) > 1: na += 1; a += g == p
    return dict(acc=ok / n, amb=a / max(na, 1), unk=au / max(nu, 1), n=n, na=na, nu=nu)
def fmt(r): return f"overall {100*r['acc']:.2f}% | ambiguous tokens {100*r['amb']:.2f}% (n={r['na']}) | unknown tokens {100*r['unk']:.2f}% (n={r['nu']})"

# ------------------------------------------------------------------ TASK 7 rule-based
H('TASK 7 - rule-based tagger')
cnt_c = Counter(min(len(c), 3) for c in dic.values())
P(f'tag dictionary from {sum(map(len,TAG_TRAIN))} training tokens: {len(dic)} word types; candidate tags per word: 1 tag={cnt_c[1]}, 2 tags={cnt_c[2]}, 3+ tags={cnt_c[3]}')
class Rule:
    def __init__(s, name, keep, pt=None, pw=None, nw=None, suf=None, cap=None, unk=None):
        s.name, s.keep, s.pt, s.pw, s.nw, s.suf, s.cap, s.unk = name, keep, pt, pw, nw, suf, cap, unk
    def fires(s, pt, pw, nw, w, unk):
        return ((s.pt is None or pt in s.pt) and (s.pw is None or pw in s.pw) and (s.nw is None or nw in s.nw) and
                (s.suf is None or w.lower().endswith(s.suf)) and (s.cap is None or w[:1].isupper() == s.cap) and (s.unk is None or unk == s.unk))
NOMINAL = {'NN', 'NNS', 'JJ', 'LOC', 'NNP', 'TIME', 'NUM'}
RULES = [
    Rule('R01 prev tag TO -> VB', {'VB'}, pt={'TO'}), Rule('R02 prev tag MD -> VB', {'VB'}, pt={'MD'}),
    Rule('R03 prev tag DT -> nominal/adjective', NOMINAL, pt={'DT'}), Rule('R04 prev tag PRP$ -> noun', {'NN', 'NNS'}, pt={'PRP$'}),
    Rule('R05 prev tag PRP -> finite verb', {'VBP', 'VBD', 'VBZ', 'MD'}, pt={'PRP'}), Rule('R06 prev tag UH (please) -> VB', {'VB'}, pt={'UH'}),
    Rule('R07 -ing after am/is -> VBG', {'VBG'}, pw={'am', 'is', 'are'}, suf=('ing',)), Rule('R08 -ed after was/is -> VBN', {'VBN'}, pw={'was', 'is', 'were'}, suf=('ed',)),
    Rule('R09 prev word from/to -> place name', {'LOC'}, pw={'from', 'to'}), Rule('R10 capitalised unknown word -> LOC/NNP', {'LOC', 'NNP'}, cap=True, unk=True),
    Rule('R11 next word fare/ticket -> noun modifier', {'NN', 'JJ'}, nw={'fare', 'ticket', 'tickets'}), Rule('R12 next word is/was -> noun/name', {'NN', 'NNS', 'LOC', 'NNP'}, nw={'is', 'was'}),
    Rule('R13 -ing suffix -> VBG', {'VBG'}, suf=('ing',)), Rule('R14 prev tag JJ -> noun', {'NN', 'NNS', 'NNP', 'LOC'}, pt={'JJ'}),
    Rule('R15 prev tag IN -> noun/place/time/det', {'LOC', 'NNP', 'TIME', 'NN', 'NNS', 'NUM', 'DT', 'PRP$', 'PRP'}, pt={'IN'}),
    Rule('R16 prev tag WRB/WP -> verb/det/adj', {'VBZ', 'VBP', 'VBD', 'MD', 'DT', 'JJ'}, pt={'WRB', 'WP'})]
def guess_cands(w):
    lw = w.lower()
    if istime(w): return {'TIME'}
    if w[:1].isupper(): return {'LOC', 'NNP'}
    if lw.endswith('ing'): return {'VBG', 'NN'}
    if lw.endswith('ed'): return {'VBN', 'VBD', 'JJ'}
    return set(OPEN)
CONFLICTS = []
def rule_tag(raw, d=dic, log=None):
    tags = []
    for i, w in enumerate(raw):
        lw = w.lower(); unk = lw not in d; cand = set(d[lw]) if not unk else guess_cands(w)
        pt = tags[-1] if tags else BOS; pw = raw[i - 1].lower() if i else BOS; nw = raw[i + 1].lower() if i + 1 < len(raw) else EOS
        fired = [(r, cand & r.keep) for r in RULES if r.fires(pt, pw, nw, w, unk) and cand & r.keep]
        c0 = cand
        for r, k in fired:
            nk = cand & r.keep
            if nk: cand = nk
        if log is not None:
            for a, b in itertools.combinations(fired, 2):
                if not (a[1] & b[1]): log.append((w, a[0].name, sorted(a[1]), b[0].name, sorted(b[1]), sorted(cand)))
        tags.append(max(cand, key=lambda t: (d[lw][t] if not unk else prior[t], t)))
    return tags
t0 = time.time(); r_rule = evaluate(lambda x: rule_tag(x, dic, CONFLICTS), TAG_TEST, dic); t_rule = time.time() - t0
r_mft = evaluate(mft_tag, TAG_TEST, dic)
P('most-frequent-tag baseline:', fmt(r_mft)); P(f'rule-based tagger ({len(RULES)} rules):', fmt(r_rule))
P('ambiguous-token accuracy is the number that matters: unambiguous words are tagged correctly by the dictionary alone, so overall accuracy hides the rules\' real contribution')
errs = Counter()
for u in TAG_TEST[:2000]:
    raw = [w for w, _ in u]
    for w, (_, g), p in zip(raw, u, rule_tag(raw)):
        if g != p: errs[(g, p)] += 1
P('five commonest rule-tagger error types (gold->predicted: count):', ', '.join(f'{g}->{p}: {c}' for (g, p), c in errs.most_common(5)))
cf = Counter((a[1], a[3]) for a in CONFLICTS)
P(f'rule conflicts (two fired rules proposing disjoint tag sets): {len(CONFLICTS)} tokens in test; commonest pair: ' + (f'{cf.most_common(1)[0][0]} x{cf.most_common(1)[0][1]}' if cf else 'none'))
if CONFLICTS: P('   example (word, rule A, A keeps, rule B, B keeps, final candidates):', CONFLICTS[0], '-> the earlier rule narrows first, the later one is ignored: ordering decides, so inserting a rule can silently change older rules')
RULES_ORIG = RULES; RULES = [RULES_ORIG[8]] + RULES_ORIG[:8] + RULES_ORIG[9:]       # experiment: R09 before R01
r_rule2 = evaluate(lambda x: rule_tag(x, dic), TAG_TEST, dic); RULES = RULES_ORIG
P(f"after moving R09 (place name after from/to) above R01: {fmt(r_rule2)}  -> one reordering changed the result; every rule must be re-tested after any edit")
PERF['rule tagger'] = dict(train_s=0.0, size_kb=deep_size(dic) / 1024, quality=f"amb {100*r_rule['amb']:.1f}%", speed=f"{r_rule['n']/t_rule:,.0f} tok/s")
PERF['MFT baseline'] = dict(train_s=0.0, size_kb=deep_size(dic) / 1024, quality=f"amb {100*r_mft['amb']:.1f}%", speed='-')

# ------------------------------------------------------------------ TASK 8 HMM + Viterbi
H('TASK 8 - HMM tagger and Viterbi')
P('Bayes: t* = argmax P(t1..tn | w1..wn) = argmax prod P(wi|ti) P(ti|ti-1)  [assumptions: word depends only on its own tag; tag depends only on previous tag]')
def feat(w):
    if istime(w): return 'time'
    lw = w.lower(); return ('C' if w[:1].isupper() else 'l') + ('ing' if lw.endswith('ing') else 'ed' if lw.endswith('ed') else 's' if lw.endswith('s') else '')
NFEAT = 9
class HMM:
    def __init__(s, sents, alpha=0.01, use_feat=True, gaz=None, cm=None):
        s.alpha, s.use_feat, s.gaz, s.cm = alpha, use_feat, gaz or set(), cm or set()
        s.tr, s.em, s.tc, s.hap, s.ht = defaultdict(Counter), defaultdict(Counter), Counter(), defaultdict(Counter), Counter()
        wc = Counter(w.lower() for x in sents for w, _ in x)
        for x in sents:
            p = BOS
            for w, t in x:
                s.tr[p][t] += 1; s.em[t][w.lower()] += 1; s.tc[t] += 1; p = t
                if wc[w.lower()] == 1: s.hap[t][feat(w)] += 1; s.ht[t] += 1
            s.tr[p][EOS] += 1
        s.tags = sorted(s.tc); s.known = defaultdict(set)
        for t in s.em:
            for w in s.em[t]: s.known[w].add(t)
        s.open = [t for t in s.tags if t in OPEN]
    def trans(s, p, t): return (s.tr[p][t] + s.alpha) / (sum(s.tr[p].values()) + s.alpha * (len(s.tags) + 1))
    def emit(s, t, w):
        lw = w.lower()
        if lw in s.known: return s.em[t][lw] / s.tc[t]
        if t == 'LOC' and lw in s.gaz: return 0.05
        if t == 'CM' and lw in s.cm: return 0.05
        return (s.hap[t][feat(w)] + 0.1) / (s.ht[t] + 0.1 * NFEAT) if s.use_feat else 1.0 / NFEAT
    def cands(s, w):
        lw = w.lower()
        if lw in s.known: return sorted(s.known[lw])
        if lw in s.gaz: return ['LOC']
        if lw in s.cm: return ['CM']
        return list(s.open) + (['CM'] if 'CM' in s.tc else [])
    def viterbi(s, raw, trace=False):
        C_ = [s.cands(w) for w in raw]; V_ = [{}]; B_ = [{}]; ops = 0
        for t in C_[0]: V_[0][t] = math.log(s.trans(BOS, t)) + math.log(s.emit(t, raw[0])); B_[0][t] = None
        for i in range(1, len(raw)):
            V_.append({}); B_.append({})
            for t in C_[i]:
                best = max(((V_[i - 1][p] + math.log(s.trans(p, t)), p) for p in C_[i - 1])); ops += len(C_[i - 1])
                V_[i][t] = best[0] + math.log(s.emit(t, raw[i])); B_[i][t] = best[1]
        last = max((V_[-1][t] + math.log(s.trans(t, EOS)), t) for t in C_[-1]); path = [last[1]]
        for i in range(len(raw) - 1, 0, -1): path.append(B_[i][path[-1]])
        path.reverse()
        return (path, last[0], V_, B_, C_, ops) if trace else path
    def exhaustive(s, raw):
        C_ = [s.cands(w) for w in raw]; best = (-1e18, None); n = 0
        for seq in itertools.product(*C_):
            lp = math.log(s.trans(BOS, seq[0])) + sum(math.log(s.trans(seq[i - 1], seq[i])) for i in range(1, len(seq))) + math.log(s.trans(seq[-1], EOS)) + sum(math.log(s.emit(t, w)) for t, w in zip(seq, raw))
            n += len(raw); best = max(best, (lp, list(seq)))
        return best, n
t0 = time.time(); hmm = HMM([u for u in TAG_TRAIN]); t_hmm_train = time.time() - t0
for p in (BOS, 'TO', 'DT'): P(f'transition row {p}: sum = {sum(hmm.trans(p,t) for t in hmm.tags)+hmm.trans(p,EOS):.6f}  (add-{hmm.alpha} smoothing; emissions are MLE for known words, hapax-suffix model for unknown words)')
t0 = time.time(); r_hmm = evaluate(hmm.viterbi, TAG_TEST, dic); t_hmm = time.time() - t0
hmm_nf = HMM(TAG_TRAIN, use_feat=False); r_hmm_nf = evaluate(hmm_nf.viterbi, TAG_TEST, dic)
P('HMM + Viterbi:', fmt(r_hmm)); P('HMM without suffix/capital evidence (open-class prior only):', fmt(r_hmm_nf))
PERF['HMM (Viterbi)'] = dict(train_s=t_hmm_train, size_kb=deep_size(hmm) / 1024, quality=f"amb {100*r_hmm['amb']:.1f}%", speed=f"{r_hmm['n']/t_hmm:,.0f} tok/s")
tr_s = max((u for u in TAG_TEST if len(u) >= 5 and all(w.lower() in dic for w, _ in u)), key=lambda u: (sum(len(dic[w.lower()]) > 1 for w, _ in u), -len(u)))
raw = [w for w, _ in tr_s]; path, lp, V_, B_, C_, ops = hmm.viterbi(raw, True); ex_best, ex_ops = hmm.exhaustive(raw)
P('Viterbi trace for:', ' '.join(raw), '| gold:', ' '.join(t for _, t in tr_s))
for i, w in enumerate(raw): P(f'  {w:9s}', '  '.join(f'{t}: {math.exp(V_[i][t]):.2e}' + (f' <-{B_[i][t]}' if B_[i][t] else '') for t in C_[i]))
P(f'best sequence: {" ".join(path)}  probability = {math.exp(lp):.3e}')
nK, nn_ = len(hmm.tags), len(raw)
P(f'operations for this sentence: Viterbi (candidate-restricted) = {ops} transitions; exhaustive (candidate-restricted) = {ex_ops} tag-steps over {len(list(itertools.product(*C_)))} sequences; unrestricted: Viterbi K^2*n = {nK**2*nn_}, exhaustive K^n*n = {nK**nn_*nn_:,}')
assert path == ex_best[1] and abs(lp - ex_best[0]) < 1e-9
P(f'accuracies on the test set: most-frequent-tag {100*r_mft["acc"]:.2f}% | HMM {100*r_hmm["acc"]:.2f}%; exhaustive search gives exactly the same tags as Viterbi (checked on this sentence)')

# ------------------------------------------------------------------ TASK 9 Brill
H('TASK 9 - transformation-based (Brill) tagger')
TNAME = ['prev tag', 'next tag', 'prev word', 'next word', 'prev two tags', 'prev tag & next tag', 'own suffix (last 3 letters)', 'capitalised']
def bctx(k, T, W, R, i):
    n = len(T); pt = T[i - 1] if i > 0 else BOS; nt = T[i + 1] if i < n - 1 else EOS
    if k == 0: return pt
    if k == 1: return nt
    if k == 2: return W[i - 1] if i > 0 else BOS
    if k == 3: return W[i + 1] if i < n - 1 else EOS
    if k == 4: return (T[i - 2] if i > 1 else BOS, pt)
    if k == 5: return (pt, nt)
    if k == 6: return W[i][-3:]
    return R[i][:1].isupper()
def bapply(rule, T, W, R):
    k, c, f, t = rule; ch = [i for i in range(len(T)) if T[i] == f and bctx(k, T, W, R, i) == c]
    for i in ch: T[i] = t
    return ch
def mk(sents): return [(R, [w.lower() for w in R], [t for _, t in u], mft_tag(R)) for u in sents for R in [[w for w, _ in u]]]
def acc_of(data): return sum(a == b for R, W, G, T in data for a, b in zip(G, T)) / sum(len(G) for R, W, G, T in data)
t0 = time.time(); btr, bdev, btest = mk(TAG_TRAIN), mk(TAG_DEV), mk(TAG_TEST)
rules_b, rlog = [], []; curve = [(acc_of(btr), acc_of(bdev), acc_of(btest))]
for it in range(60):
    good, corr = Counter(), Counter()
    for R, W, G, T in btr:
        for i in range(len(T)):
            f = T[i]
            for k in range(8):
                c = bctx(k, T, W, R, i)
                if f == G[i]: corr[(k, c, f)] += 1
                else: good[(k, c, f, G[i])] += 1
    if not good: break
    best = max(good, key=lambda r: good[r] - corr[(r[0], r[1], r[2])])
    if good[best] - corr[(best[0], best[1], best[2])] < 1: break
    e_before = sum(a != b for R, W, G, T in btr for a, b in zip(G, T)); cor = intro = 0
    for D in (btr, bdev, btest):
        for R, W, G, T in D:
            for i in bapply(best, T, W, R):
                if D is btr: cor += best[3] == G[i]; intro += best[2] == G[i]
    e_after = sum(a != b for R, W, G, T in btr for a, b in zip(G, T)); rules_b.append(best); rlog.append((cor, intro, e_before - e_after)); curve.append((acc_of(btr), acc_of(bdev), acc_of(btest)))
t_brill = time.time() - t0
P(f'initial state = most-frequent tag (train acc {100*curve[0][0]:.2f}%); templates: ' + '; '.join(TNAME) + ' (tag window -2..+1, word window -1..+1: wider windows add many rare contexts but this grammar is local)')
P('first ten transformations (rule | corrected | introduced):')
for r, (c, i_, d_) in list(zip(rules_b, rlog))[:10]: P(f'  change {r[2]} -> {r[3]} if {TNAME[r[0]]} = {r[1]}   | +{c} | -{i_}')
if rlog: assert rlog[0][2] == rlog[0][0] - rlog[0][1]
devacc = [c[1] for c in curve]; bpt = devacc.index(max(devacc))
P(f'accuracy after k rules (dev): k=0 {100*devacc[0]:.2f}%, k=5 {100*devacc[min(5,len(devacc)-1)]:.2f}%, k=10 {100*devacc[min(10,len(devacc)-1)]:.2f}%, k=all({len(rules_b)}) {100*devacc[-1]:.2f}%; dev accuracy peaks at k={bpt}; test accuracy there {100*curve[bpt][2]:.2f}%, with all rules {100*curve[-1][2]:.2f}%; train accuracy keeps rising to {100*curve[-1][0]:.2f}%')
P('rules that name one specific word or a rare suffix (e.g. "prev word = ...", "suffix = ...") are artefacts of the training data; rules on prev tag = TO/DT/PRP$ are linguistic generalisations')
for r in rules_b:
    if r[0] in (2, 3, 6): P('  artefact-type rule found:', f'change {r[2]} -> {r[3]} if {TNAME[r[0]]} = {r[1]}'); break
def brill_tag_factory():
    def f(raw):
        W = [w.lower() for w in raw]; T = mft_tag(raw)
        for r in rules_b[:bpt] if False else rules_b: bapply(r, T, W, raw)
        return T
    return f
t0 = time.time(); r_brill = evaluate(brill_tag_factory(), TAG_TEST, dic); t_b = time.time() - t0
P('Brill tagger on test:', fmt(r_brill))
PERF['Brill (TBL)'] = dict(train_s=t_brill, size_kb=deep_size(rules_b) / 1024, quality=f"amb {100*r_brill['amb']:.1f}%", speed=f"{r_brill['n']/t_b:,.0f} tok/s")
if plt:
    plt.figure(figsize=(5, 3.6)); plt.plot([100 * c[0] for c in curve], label='train'); plt.plot([100 * c[1] for c in curve], label='dev'); plt.axvline(bpt, ls=':', c='gray')
    plt.xlabel('number of transformations'); plt.ylabel('accuracy %'); plt.title('Brill learning curve'); plt.legend(); plt.tight_layout(); plt.savefig('plots/brill_curve.png', dpi=130); plt.close()

# ================================================================== PART 3: TASK 10 (results), TASK 12 (design challenge), TASK 11 (effort)
H('TASK 10 - learning curve and plots')
LC = {'MFT': [], 'HMM': []}; sizes = [100, 250, 500, 1000, 2500, 5000]
for n in sizes:
    sub = TAG_TRAIN[:n]; d = defaultdict(Counter)
    for u in sub:
        for w, t in u: d[w.lower()][t] += 1
    LC['MFT'].append(evaluate(make_mft(d), TAG_TEST, d)['acc']); LC['HMM'].append(evaluate(HMM(sub).viterbi, TAG_TEST, d)['acc'])
P('tagging accuracy (test) vs number of training utterances', sizes, '| MFT', [round(100 * a, 2) for a in LC['MFT']], '| HMM', [round(100 * a, 2) for a in LC['HMM']])
if plt:
    plt.figure(figsize=(5, 3.6)); plt.plot(sizes, [100 * a for a in LC['MFT']], 'o-', label='most-frequent tag'); plt.plot(sizes, [100 * a for a in LC['HMM']], 's-', label='HMM'); plt.xscale('log')
    plt.xlabel('training utterances'); plt.ylabel('test accuracy %'); plt.legend(); plt.tight_layout(); plt.savefig('plots/accuracy_vs_size.png', dpi=130); plt.close()
    plt.figure(figsize=(5, 3.6)); plt.bar(['unigram', 'bigram Katz', 'trigram Katz'], [RES['unigram MLE'][0], RES['bigram Katz/Good-Turing'][0], RES['trigram Katz backoff'][0]]); plt.yscale('log')
    plt.ylabel('dev perplexity'); plt.title('perplexity vs n-gram order'); plt.tight_layout(); plt.savefig('plots/perplexity_vs_order.png', dpi=130); plt.close()

H('TASK 12 - changed requirement: rail enquiries and code-mixed speech')
rn = random.Random(SEED + 7); new_locs = NEW_LOCS + OLD_LOCS[:50]; new_w = [0.6 / len(NEW_LOCS)] * len(NEW_LOCS) + [0.4 / 50] * 50
NEWU = [make_utt(rn, new_locs, new_w, new=True) for _ in range(300)]; ADAPT, EVAL = NEWU[:150], NEWU[150:]
adapt_lm, eval_lm = [lm(u) for u in ADAPT], [lm(u) for u in EVAL]
oov = lambda sents: sum(1 for x in sents for w in x if w not in trc) / sum(map(len, sents))
ev_tok = [w for x in eval_lm for w in x]
P(f'A. new-call OOV rate = {100*oov(eval_lm):.1f}% (was {100*dev_oov_full:.2f}% on dev). Of the unseen tokens: unseen station names {sum(1 for w in ev_tok if w in LOCSET and w not in trc)}, code-mixed words {sum(1 for w in ev_tok if w in CMW and w not in trc)}, new transport words {sum(1 for w in ev_tok if w in ("metro","train","intercity") and w not in trc)} of {sum(1 for w in ev_tok if w not in trc)}')
P('   breaks: closed vocabulary (UNK is no longer rare), stable OOV rate, English-only classes (no tag for code-mixed words), transition statistics from a bus-only domain; the lost words are mostly the place names the enquiry depends on')
gaz = set(LOCSET); cmlex = {w.lower() for u in ADAPT for w, t in u if t == 'CM'}
bld = lambda sm: (lambda ix: Interp(Counts(sm), em_lambdas(Counts([sm[i] for i in ix[:int(.9 * len(ix))]]), [sm[i] for i in ix[int(.9 * len(ix)):]])[0]))(random.Random(SEED).sample(range(len(sm)), len(sm)))
L2 = math.log2
cl_cnt = Counter(w for x in train_lm + adapt_lm for w in x if w in gaz); cl_tot = sum(cl_cnt.values()); cm_cnt = Counter(w for x in adapt_lm for w in x if w in cmlex); cm_tot = sum(cm_cnt.values())
def sp_plain(tok, sym): return L2(10000) if sym == UNK else 0.0                     # a hidden word still has to be named: 1/10000 of the UNK mass
def sp_class(tok, sym):
    if sym == '<LOC>': return -L2((cl_cnt[tok] + 1) / (cl_tot + len(gaz)))
    if sym == '<CM>': return -L2((cm_cnt[tok] + 1) / (cm_tot + len(cmlex)))
    return L2(10000) if sym == UNK else 0.0
def adj_bits(model, sents, mapper, spread, total=False):
    bits = n = 0
    for toks in sents:
        q = [BOS, BOS] + mapper(toks) + [EOS]
        for i in range(2, len(q)):
            bits -= L2(max(model.p(q[i], q[i - 2], q[i - 1]), 1e-12)); n += 1
            if i - 2 < len(toks): bits += spread(toks[i - 2], q[i])
    return bits if total else bits / n
VOCr = {w for w, c in Counter(w for x in train_lm + adapt_lm for w in x).items() if c > 1}
mp_orig = lambda t: mapu(t, VOC); mp_retr = lambda t: mapu(t, VOCr)
mp_cls = lambda t: ['<LOC>' if w in gaz else '<CM>' if w in cmlex else w for w in t]
cls_train = [mp_cls(x) for x in train_lm + adapt_lm]; VOCc = {w for w, c in Counter(w for x in cls_train for w in x).items() if c > 1} | {'<LOC>', '<CM>'}
mp_red = lambda t: mapu(mp_cls(t), VOCc)
m_orig = interp; m_retr = bld([mp_retr(x) for x in train_lm + adapt_lm]); m_red = bld([mapu(x, VOCc) for x in cls_train])
P('\nB/C. language model (adjusted perplexity = every hidden word is charged for being named, so models are comparable)')
P('model                                             old test PPL   new-calls PPL')
LMS = [('original (retain)', m_orig, mp_orig, sp_plain), ('retrained on train+150 new calls (modify)', m_retr, mp_retr, sp_plain), ('redesign: station gazetteer + <LOC>/<CM> classes', m_red, mp_red, sp_class)]
for nm, m, mp, sp in LMS: P(f'{nm:50s}{2**adj_bits(m, test_lm, mp, sp):10.2f}{2**adj_bits(m, eval_lm, mp, sp):14.2f}')
nsets = make_sets(eval_lm, random.Random(SEED + 3), 100); wq = RESC['trigram deleted interp'][0]
w_before = wer(nsets, lambda h: max(h, key=lambda x: x[1])[0])
P(f'rescoring 100 new-call hypothesis sets: WER of recogniser 1-best {100*w_before:.2f}%; after rescoring (LM weight {wq}):')
for nm, m, mp, sp in LMS: P(f'   {nm:50s}{100*wer(nsets, best_by(nsets, lambda t, m=m, mp=mp, sp=sp: -adj_bits(m, [t], mp, sp, True), wq)):6.2f}%')
# traces
def pick(f): return next(x for x in eval_lm if f(x))
tr3 = [pick(lambda x: any(w in LOCSET and w not in trc for w in x) and not any(w in CMW for w in x)), pick(lambda x: any(w in CMW for w in x) and not any(w in LOCSET and w not in trc for w in x)),
       pick(lambda x: any(w in LOCSET and w not in trc for w in x) and any(w in CMW for w in x))]
P('traces through the ORIGINAL LM and HMM tagger (what fails first):')
for x in tr3:
    q = [BOS, BOS] + mp_orig(x) + [EOS]; ps = [(q[i], model_p) for i in range(2, len(q)) for model_p in [m_orig.p(q[i], q[i - 2], q[i - 1])]]
    raw = [w for u in EVAL if lm(u) == x for w, _ in u][:len(x)]; gold = next([t for _, t in u] for u in EVAL if lm(u) == x); pr = hmm.viterbi(raw)
    P('  ', ' '.join(x)); P('     LM:', ' '.join(f'{w}:{p:.3f}' for w, p in ps if w != EOS)); P('     tags gold/pred:', ' '.join(f'{g}/{p}' for g, p in zip(gold, pr)))
P('\ntagging on new calls (accuracy; code-mixed tokens carry gold tag CM which the original tagset does not have):')
dic_new = defaultdict(Counter)
for u in TAG_TRAIN + ADAPT:
    for w, t in u: dic_new[w.lower()][t] += 1
hmm_new = HMM(TAG_TRAIN + ADAPT, gaz=gaz, cm=cmlex)
for nm, fn, d in (('most-frequent tag', mft_tag, dic), ('rule-based', lambda x: rule_tag(x, dic), dic), ('HMM original', hmm.viterbi, dic), ('Brill original', brill_tag_factory(), dic), ('HMM redesigned (+CM tag, gazetteer, retrained)', hmm_new.viterbi, dic_new)):
    r = evaluate(fn, EVAL, d); cmacc = sum(1 for u in EVAL for (w, g), p in zip(u, fn([w for w, _ in u])) if g == 'CM' and p == 'CM') / max(1, sum(1 for u in EVAL for _, g in u if g == 'CM'))
    P(f'   {nm:48s} overall {100*r["acc"]:.2f}% | unknown tokens {100*r["unk"]:.2f}% (n={r["nu"]}) | code-mixed tokens {100*cmacc:.1f}%')
r_old = evaluate(hmm_new.viterbi, TAG_TEST, dic_new); P('   redesigned HMM on the OLD test set (no regression check):', fmt(r_old))
P('decision: MODIFY + partial REDESIGN - keep the n-gram/HMM framework; add gazetteer + class-based n-gram, <CM> tag, retrain with new calls; refresh gazetteer daily, re-estimate lambdas and retrain monthly')
PERF['HMM redesigned'] = dict(train_s=0.0, size_kb=deep_size(hmm_new) / 1024, quality=f"new calls {100*evaluate(hmm_new.viterbi, EVAL, dic_new)['acc']:.1f}%", speed='-')

H('TASK 10 - reproducible tests (hand calculation vs program)')
tests = [('zero-probability utterance (Task 2): bigram MLE = 0, add-k > 0', hand(ZERO_UTT)[0] == 0 and xent(AddK(C, 2, bestk), [ZERO_UTT])[2] == 0),
         ('lambda weights (Task 4) sum to 1 in every band', all(abs(sum(l) - 1) < 1e-9 for l in lam.values())),
         ('Katz and interpolated models are valid distributions (Task 4)', abs(sum(katz3.p(w, BOS, BOS) for w in C.u) - 1) < 1e-6 and abs(sum(interp.p(w, BOS, BOS) for w in C.u) - 1) < 1e-6),
         ('Viterbi = exhaustive search; trellis best path (Task 8)', path == ex_best[1]),
         ('first Brill rule: corrected - introduced = drop in training errors (Task 9)', rlog[0][0] - rlog[0][1] == rlog[0][2])]
for nm, ok in tests: P(('PASS ' if ok else 'FAIL ') + nm)
assert all(ok for _, ok in tests)
P('\nmodel / tagger                      train time(s)  size(KB)   quality                 speed')
for nm, d in PERF.items(): P(f"{nm:34s}{d.get('train_s', 0):9.2f}  {d.get('size_kb', 0):9.0f}   {d.get('quality', '-'):22s}  {d.get('speed', '-')}")

H('TASK 11 - effort estimate (planning assumptions for a real deployment)')
P('transcription 120 utt/h x 20,000 = 167 h; annotation 400 tokens/h: 3,000 tokens = 7.5 h + 500 doubly tagged = 1.25 h; tagset design 4 h; 16 rules incl. testing ~ 8 h; adaptation sample (150 calls, ~1,300 tokens) = 3.3 h per refresh')
P('at an assumed Rs 300/h: annotation+rules ~ Rs 6,000 once, ~Rs 1,000 per monthly refresh; the HMM gains most from more annotated data, rules stop paying once ambiguity left is lexical (not contextual)')
LOG.close()


========== TASK 1 - counting words, Zipf, vocabulary ==========
corpus: 24000 utterances, 177178 tokens | split 19194/2401/2405 utterances (140776/17879/18523 tokens)
tokens=177178  word forms (case-sensitive)=1342  types (case-folded)=1001  lemmas=994  (<s>,</s> added per utterance, not counted as words)
decision 1 case-folding: vocabulary 1342 -> 1001 types | decision 2 "6:30" as 1 token vs 3 tokens: vocabulary 1001 vs 985, tokens 177178 vs 182238
Zipf: log f = 13.61 -1.97 log(rank), R^2=0.975 (ideal slope -1)
largest deviations (residual, word, rank, freq): below line [(-3.88, 'to', 1, 16845), (-2.85, 'the', 2, 12082), (-2.2, 'bus', 3, 10387)] | above line [(0.78, 'or', 51, 761), (0.81, 'will', 52, 755), (0.85, 'next', 53, 755)]
threshold 1 (count<=1 -> <UNK>): vocabulary=746 types, 0.15% of training tokens become <UNK>
threshold 2 (count<=2 -> <UNK>): vocabulary=577 types, 0.39% of training tokens become <UNK>
threshold 3 (count<=3 -> <UNK>): vocabulary=457 types, 0.65% of trainin